# Low-Entropy LLM Watermarking

**Baseline**: Soft red-list watermark from Kirchenbauer et al. (ICML 2023)  
Paper: https://arxiv.org/abs/2301.10226  
Original code: https://github.com/jwkirchenbauer/lm-watermarking

This notebook is the interactive entry-point. The core logic lives in:
- `watermark.py` — `WatermarkLogitsProcessor` + `WatermarkDetector`
- `generate.py` — bulk generation on C4 RealNewsLike
- `evaluate.py` — z-scores, PPL, ROC/AUC

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from watermark import WatermarkLogitsProcessor, WatermarkDetector

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "facebook/opt-1.3b"
GAMMA = 0.25
DELTA = 2.0
HASH_KEY = 15485863

print(f"Using device: {DEVICE}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
).to(DEVICE)
model.eval()
print("Model loaded.")

In [ ]:
vocab_size = len(tokenizer)

watermark_processor = WatermarkLogitsProcessor(
    vocab_size=vocab_size,
    gamma=GAMMA,
    delta=DELTA,
    hash_key=HASH_KEY,
)

detector = WatermarkDetector(
    vocab_size=vocab_size,
    gamma=GAMMA,
    hash_key=HASH_KEY,
    z_threshold=4.0,
)

## Quick demo: single prompt

In [ ]:
PROMPT = (
    "The quick brown fox jumps over the lazy dog. "
    "In recent news, scientists have discovered"
)

@torch.inference_mode()
def generate(prompt: str, watermarked: bool, max_new_tokens: int = 200) -> str:
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(DEVICE)
    processors = [watermark_processor] if watermarked else []
    out = model.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        logits_processor=processors,
    )
    new_tokens = out[0, input_ids.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True), new_tokens.tolist()

nw_text, nw_tokens = generate(PROMPT, watermarked=False)
w_text,  w_tokens  = generate(PROMPT, watermarked=True)

print("=== No watermark ===")
print(nw_text[:300])
print()
print("=== Watermarked ===")
print(w_text[:300])

In [ ]:
nw_result = detector.detect(nw_tokens)
w_result  = detector.detect(w_tokens)

print("No-watermark detection:", nw_result)
print("Watermarked detection: ", w_result)

## Bulk generation on C4

Run from terminal (slow — downloads model + dataset):
```bash
python generate.py --num_samples 500 --output_file data/generations.jsonl
```

Then evaluate:
```bash
python evaluate.py --input_file data/generations.jsonl
# add --compute_ppl to also measure oracle perplexity (requires OPT-2.7B)
```

---
## Extensions: Adaptive δ and Forced Generation

Two extensions beyond the Kirchenbauer et al. baseline:
1. **Adaptive δ** — scales the green-list bias with the Shannon entropy of the current distribution. Low-entropy (near-deterministic) steps get δ ≈ 0; high-entropy steps get the full δ.
2. **Forced generation** — continues generating token-by-token until the running z-score exceeds the detection threshold, guaranteeing detectability at the cost of variable output length.


In [ ]:
# Adaptive processor and detector (fixed processor already set up above)
adaptive_processor = WatermarkLogitsProcessor(
    vocab_size=vocab_size,
    gamma=GAMMA,
    delta=DELTA,
    adaptive=True,
    hash_key=HASH_KEY,
)
detector = WatermarkDetector(
    vocab_size=vocab_size,
    gamma=GAMMA,
    hash_key=HASH_KEY,
    z_threshold=4.0,
)
print('Processors and detector ready.')


### Per-token entropy: fixed δ vs adaptive δ

The cell below wraps the logits processor to log the normalized entropy and applied δ at each generation step, then plots them side by side. The key claim: in low-entropy positions the adaptive version applies near-zero bias, while the fixed version always applies the full δ regardless of how predictable the next token is.

In [ ]:
import math, numpy as np, matplotlib.pyplot as plt

class EntropyLoggingProcessor(WatermarkLogitsProcessor):
    """Thin wrapper that records entropy and applied delta at each step."""
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.entropy_log = []
        self.delta_log   = []

    def __call__(self, input_ids, scores):
        norm_h = self._normalized_entropy(scores[0])
        applied = self._adaptive_delta(scores[0]) if self.adaptive else self.delta
        self.entropy_log.append(float(norm_h))
        self.delta_log.append(float(applied))
        return super().__call__(input_ids, scores)

N_DEMO = 100  # tokens to generate for the entropy demo
LOW_ENTROPY_PROMPT = (
    "The capital of France is Paris. The capital of Germany is Berlin. "
    "The capital of Italy is Rome. The capital of Spain is"
)

fixed_logger    = EntropyLoggingProcessor(vocab_size=vocab_size, gamma=GAMMA, delta=DELTA, adaptive=False, hash_key=HASH_KEY)
adaptive_logger = EntropyLoggingProcessor(vocab_size=vocab_size, gamma=GAMMA, delta=DELTA, adaptive=True,  hash_key=HASH_KEY)

@torch.inference_mode()
def generate_logged(prompt, proc, n=N_DEMO):
    ids = tokenizer.encode(prompt, return_tensors='pt').to(DEVICE)
    out = model.generate(ids, max_new_tokens=n, do_sample=True, logits_processor=[proc])
    return tokenizer.decode(out[0, ids.shape[1]:], skip_special_tokens=True)

fixed_text    = generate_logged(LOW_ENTROPY_PROMPT, fixed_logger)
adaptive_text = generate_logged(LOW_ENTROPY_PROMPT, adaptive_logger)
print('Fixed:   ', fixed_text[:120])
print('Adaptive:', adaptive_text[:120])


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 7))

axes[0].plot(fixed_logger.entropy_log,    label='Fixed δ',    color='steelblue')
axes[0].plot(adaptive_logger.entropy_log, label='Adaptive δ', color='darkorange', linestyle='--')
axes[0].set_ylabel('Normalized\nEntropy')
axes[0].axhline(0.5, color='gray', linewidth=0.8, linestyle=':')
axes[0].legend()
axes[0].set_title('Per-token normalized entropy (fixed δ vs adaptive δ)')

axes[1].plot(fixed_logger.delta_log,    color='steelblue',  label='Fixed δ applied')
axes[1].plot(adaptive_logger.delta_log, color='darkorange', linestyle='--', label='Adaptive δ applied')
axes[1].set_ylabel('δ applied')
axes[1].legend()

adp_steps = range(len(adaptive_logger.delta_log))
axes[2].fill_between(adp_steps, adaptive_logger.delta_log, DELTA,
                     alpha=0.3, color='green', label='δ saved (low-entropy positions)')
axes[2].set_ylabel('δ reduction')
axes[2].set_xlabel('Token step')
axes[2].legend()

plt.tight_layout()
plt.savefig('entropy_vs_delta.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Fixed length: {len(fixed_logger.entropy_log)} tokens  |  Adaptive: {len(adaptive_logger.entropy_log)} tokens')
print(f'Avg entropy — Fixed: {np.mean(fixed_logger.entropy_log):.3f}  Adaptive: {np.mean(adaptive_logger.entropy_log):.3f}')
print(f'Avg δ applied — Fixed: {np.mean(fixed_logger.delta_log):.3f}  Adaptive: {np.mean(adaptive_logger.delta_log):.3f}')


### Forced generation: z-score trajectory

Forced generation runs a token-by-token loop, checking the running z-score after each token. The plot below shows how the z-score grows and where it crosses the detection threshold.

In [ ]:
from generate import generate_until_detected

forced_ids = generate_until_detected(
    model, tokenizer.encode(PROMPT, return_tensors='pt')[0].tolist(),
    watermark_processor, detector,
    max_tokens=800, eos_token_id=tokenizer.eos_token_id, device=DEVICE,
)

z_trajectory = [
    detector.detect(forced_ids[:t+1])['z_score']
    for t in range(1, len(forced_ids))
]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(z_trajectory, color='steelblue', label='Running z-score')
ax.axhline(4.0, color='red', linestyle='--', linewidth=1.2, label='Detection threshold (z=4.0)')
ax.axhline(0,   color='gray', linewidth=0.5)
ax.fill_between(range(len(z_trajectory)), z_trajectory, 4.0,
                where=[z > 4.0 for z in z_trajectory], alpha=0.2, color='green', label='Detected')
ax.set_xlabel('Tokens generated')
ax.set_ylabel('Z-score')
ax.set_title(f'Forced generation z-score trajectory  (stopped at {len(forced_ids)} tokens)')
ax.legend()
plt.tight_layout()
plt.savefig('forced_trajectory.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Forced length: {len(forced_ids)} tokens  |  final z: {z_trajectory[-1]:.2f}')


### Low-entropy scenario: detection comparison

Run all four conditions on a structured, low-entropy prompt and compare z-scores and detection. Low-entropy prompts (lists, fill-in-the-blank) are where adaptive δ and forced generation are most relevant.

In [ ]:
STRUCTURED_PROMPT = (
    "Country: France  Capital: Paris\n"
    "Country: Germany Capital: Berlin\n"
    "Country: Japan   Capital: Tokyo\n"
    "Country: Brazil  Capital:"
)
N_STRUCT = 80

@torch.inference_mode()
def gen(prompt, proc=None, n=N_STRUCT):
    ids = tokenizer.encode(prompt, return_tensors='pt').to(DEVICE)
    out = model.generate(ids, max_new_tokens=n, do_sample=True,
                         logits_processor=[proc] if proc else [])
    return out[0, ids.shape[1]:].tolist()

prompt_ids_struct = tokenizer.encode(STRUCTURED_PROMPT, return_tensors='pt')[0].tolist()

nw_s         = gen(STRUCTURED_PROMPT)
fixed_s      = gen(STRUCTURED_PROMPT, watermark_processor)
adp_s        = gen(STRUCTURED_PROMPT, adaptive_processor)
forced_s     = generate_until_detected(model, prompt_ids_struct, watermark_processor, detector,
                                        max_tokens=400, eos_token_id=tokenizer.eos_token_id, device=DEVICE)
forced_adp_s = generate_until_detected(model, prompt_ids_struct, adaptive_processor, detector,
                                        max_tokens=400, eos_token_id=tokenizer.eos_token_id, device=DEVICE)

rows = [
    ('No watermark',    nw_s),
    ('Fixed δ',         fixed_s),
    ('Adaptive δ',      adp_s),
    ('Forced (fixed)',  forced_s),
    ('Forced (adapt.)', forced_adp_s),
]

print(f'{"Condition":<16}  {"Tokens":>6}  {"Z-score":>8}  {"Detected":>9}  Text preview')
print('-' * 82)
for label, toks in rows:
    r = detector.detect(toks)
    preview = tokenizer.decode(toks[:20], skip_special_tokens=True).replace('\n',' ')
    print(f'{label:<16}  {len(toks):>6}  {r["z_score"]:>8.2f}  {str(r["is_watermarked"]):>9}  {preview}')


### Bulk evaluation results

After running `generate.py` and `evaluate.py`, load the saved summary and plot the four conditions side by side.
```bash
!python generate.py --num_samples 500 --output_file data/generations.jsonl
!python evaluate.py --input_file data/generations.jsonl
```

In [ ]:
# ── Low-entropy performance breakdown (z-score + perplexity) ──────────────────
import json, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from watermark import WatermarkDetector

records   = [json.loads(l) for l in Path('data/generations.jsonl').read_text().strip().split('\n')]
eval_path = Path('data/generations.eval.json')
ev        = json.loads(eval_path.read_text()) if eval_path.exists() else {}

detector = WatermarkDetector(vocab_size=vocab_size, gamma=GAMMA, hash_key=HASH_KEY, z_threshold=4.0)

def seq_entropy(toks):
    counts = np.bincount(toks)
    p = counts[counts > 0] / len(toks)
    return float(-np.sum(p * np.log(p)))

entropies = np.array([seq_entropy(r['no_watermark_tokens']) for r in records])
median_h  = np.median(entropies)
low_mask  = entropies < median_h
high_mask = ~low_mask

def score(recs, mask, tok_key, ppl_list=None):
    idx  = [i for i, m in enumerate(mask) if m and tok_key in recs[i]]
    zs   = [detector.detect(recs[i][tok_key])['z_score'] for i in idx]
    lens = [len(recs[i][tok_key]) for i in idx]
    ppls = [ppl_list[i] for i in idx] if ppl_list else None
    return dict(tpr=np.mean([z > 4.0 for z in zs]),
                z_mean=np.mean(zs),
                len_mean=np.mean(lens),
                ppl_mean=np.nanmean(ppls) if ppls else None)

conditions = {
    'Fixed δ':         ('watermarked_tokens',  ev.get('w_ppls')),
    'Adaptive δ':      ('adaptive_tokens',     ev.get('adp_ppls')),
    'Forced (fixed)':  ('forced_tokens',       ev.get('forced_ppls')),
    'Forced (adapt.)': ('forced_adp_tokens',   ev.get('forced_adp_ppls')),
}

stats = {}
for split, mask in [('Low', low_mask), ('High', high_mask)]:
    stats[split] = {}
    for cond, (tok_key, ppl_list) in conditions.items():
        if tok_key in records[0]:
            stats[split][cond] = score(records, mask, tok_key, ppl_list)

conds   = list(stats['Low'].keys())
has_ppl = any(stats['Low'][c]['ppl_mean'] is not None for c in conds)

# ── Print table ────────────────────────────────────────────────────────────────
ppl_h = f"{'PPL':>7}" if has_ppl else ''
w_col = 42 if has_ppl else 28
print(f"\n{'':16} {'── Low entropy ──':^{w_col}} {'── High entropy ──':^{w_col}}")
print(f"{'':16} {'TPR':>7} {'Z-mean':>7} {'Len':>6}{' '+ppl_h if has_ppl else ''}   {'TPR':>7} {'Z-mean':>7} {'Len':>6}{' '+ppl_h if has_ppl else ''}")
print('─' * (95 if has_ppl else 76))
for c in conds:
    lo, hi = stats['Low'][c], stats['High'][c]
    ppl_lo = f"  {lo['ppl_mean']:>6.2f}" if has_ppl and lo['ppl_mean'] else ''
    ppl_hi = f"  {hi['ppl_mean']:>6.2f}" if has_ppl and hi['ppl_mean'] else ''
    print(f"{c:16} {lo['tpr']:>7.3f} {lo['z_mean']:>7.2f} {lo['len_mean']:>6.1f}{ppl_lo}"
          f"   {hi['tpr']:>7.3f} {hi['z_mean']:>7.2f} {hi['len_mean']:>6.1f}{ppl_hi}")

# ── Plots ──────────────────────────────────────────────────────────────────────
n_plots = 4 if has_ppl else 3
fig, axes = plt.subplots(1, n_plots, figsize=(5*n_plots, 4))
x, w = np.arange(len(conds)), 0.35

metrics = [
    ('tpr',      'TPR',                  'Detection rate (TPR @ z>4.0)'),
    ('z_mean',   'Mean z-score',         'Mean z-score'),
    ('len_mean', 'Tokens generated',     'Output length'),
]
if has_ppl:
    metrics.append(('ppl_mean', 'Perplexity (↓ better)', 'Oracle perplexity (OPT-2.7B)'))

for ax, (metric, ylabel, title) in zip(axes, metrics):
    lo_vals = [stats['Low'][c][metric]  or 0 for c in conds]
    hi_vals = [stats['High'][c][metric] or 0 for c in conds]
    ax.bar(x - w/2, lo_vals, w, label='Low entropy',  color='steelblue')
    ax.bar(x + w/2, hi_vals, w, label='High entropy', color='darkorange')
    ax.set_xticks(x); ax.set_xticklabels(conds, rotation=15)
    ax.set_ylabel(ylabel); ax.set_title(title)
    if metric == 'tpr': ax.set_ylim(0, 1.05)
axes[0].legend()

plt.suptitle('Low- vs high-entropy breakdown  |  OPT-1.3B on C4 RealNewsLike\n'
             '(median split on unigram entropy of no-watermark output)', fontsize=10)
plt.tight_layout()
plt.savefig('entropy_split.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Conclusions ────────────────────────────────────────────────────────────────
nw_ppl = ev.get('nw_ppl_mean')
print('\n── Conclusions ──')
for c in conds:
    lo, hi = stats['Low'][c], stats['High'][c]
    print(f"\n  {c}:")
    print(f"    TPR   low={lo['tpr']:.3f}  high={hi['tpr']:.3f}  drop={lo['tpr']-hi['tpr']:+.3f}")
    print(f"    Z     low={lo['z_mean']:.2f}  high={hi['z_mean']:.2f}")
    if lo['ppl_mean']:
        ppl_vs_nw_lo = lo['ppl_mean'] - nw_ppl if nw_ppl else 0
        ppl_vs_nw_hi = hi['ppl_mean'] - nw_ppl if nw_ppl else 0
        print(f"    PPL   low={lo['ppl_mean']:.2f} ({ppl_vs_nw_lo:+.2f} vs no-wm)  "
              f"high={hi['ppl_mean']:.2f} ({ppl_vs_nw_hi:+.2f} vs no-wm)")
    print(f"    Len   low={lo['len_mean']:.1f}  high={hi['len_mean']:.1f} tokens")
